### 0. 주가 예측을 위한 범위 설정

In [10]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [11]:
ticker = 'NVDA' # 주식명

start_date = '2020-01-01' # 기간
end_date = '2024-09-30'

stock_df = yf.download(ticker, start = start_date, end = end_date)

[*********************100%***********************]  1 of 1 completed


### 1. 보조 지표 추가

In [12]:
import ta

In [13]:
# Series로 변환
adj_close_series = stock_df['Adj Close'].squeeze()
volume_series = stock_df['Volume'].squeeze()

## 1.1. 단순 이동평균 (Simple Moving Average)
stock_df['MA5'] = adj_close_series.rolling(window=5).mean()        # 5일 SMA
stock_df['MA20'] = adj_close_series.rolling(window=20).mean()      # 20일 SMA
stock_df['MA60'] = adj_close_series.rolling(window=60).mean()      # 60일 SMA
stock_df['MA120'] = adj_close_series.rolling(window=120).mean()    # 120일 SMA

## 1.2. 지수 이동 평균 (Exponential Moving Average)
stock_df['EMA5'] = adj_close_series.ewm(span=5, adjust=False).mean()       # 5일 EMA
stock_df['EMA20'] = adj_close_series.ewm(span=20, adjust=False).mean()     # 20일 EMA
stock_df['EMA60'] = adj_close_series.ewm(span=60, adjust=False).mean()     # 60일 EMA
stock_df['EMA120'] = adj_close_series.ewm(span=120, adjust=False).mean()   # 120일 EMA

## 1.3. 더블 볼린저 밴드 (Double Bollinger Bands indicator)

# 중앙선 (Central Line : 20-day moving average)
stock_df['BOL_AVG'] = ta.volatility.bollinger_mavg(adj_close_series)

# 더블 볼린저 밴드 (Double Bollinger Bands)
rolling_std = adj_close_series.rolling(window=20).std()
stock_df['BOL_H1'] = stock_df['BOL_AVG'] + 2 * rolling_std
stock_df['BOL_L1'] = stock_df['BOL_AVG'] - 2 * rolling_std
stock_df['BOL_H2'] = stock_df['BOL_AVG'] + rolling_std
stock_df['BOL_L2'] = stock_df['BOL_AVG'] - rolling_std

## 1.4. RSI(Relative Strength Index)
stock_df['RSI'] = ta.momentum.rsi(adj_close_series)

## 1.5. MACD (Moving Average Convergence Divergence )
stock_df['MACD'] = ta.trend.macd(adj_close_series)
stock_df['MACD_SIGNAL']= ta.trend.macd_signal(adj_close_series)

## 1.6. OBV (On-Balance Volume)
stock_df['OBV'] = ta.volume.on_balance_volume(adj_close_series, volume_series)

### 2. 경제지표 불러오기

In [14]:
from fredapi import Fred
import FinanceDataReader as fdr

In [15]:
adj_close_df = stock_df[['Adj Close']]

## 2.1. 미국 국채 수익률 (U.S. Treasury bond yields) 
fred = Fred(api_key = '4c55d0ee6170369793707da4cba1b7be')
dgs2 = fred.get_series('DGS2', observation_start=start_date, observation_end=end_date)      # 2년 DGS
dgs5 = fred.get_series('DGS5', observation_start=start_date, observation_end=end_date)      # 5년 DGS
dgs10 = fred.get_series('DGS10', observation_start=start_date, observation_end=end_date)    # 10년 DGS

DGS = pd.concat([dgs2, dgs5,dgs10], axis=1)
DGS.columns = ['2-year', '5-year', '10-year']
DGS.index.name = 'Date'

## 2.2. 미국 국채 장단기 금리 차 (U.S. long-short interest rate spread (10-year & 2-year))
T10Y2Y = fdr.DataReader('FRED:T10Y2Y', start_date, end_date)
T10Y2Y.index.name = 'Date'

## 2.3. 변동성 지수 (Volatility Index : VIX)
VIX = fdr.DataReader('FRED:VIXCLS', start_date, end_date)
VIX.index.name = 'Date'

## 2.4. 실업률 (Unemployment Rate)
Unemployment_Rate = fdr.DataReader('FRED:UNRATE', start_date, end_date)
Unemployment_Rate.index.name = 'Date'

## 2.5. 소비자 물가 지수 (Consumer Price Index : CPI)
CPI = fdr.DataReader('FRED:CPIAUCSL', start_date, end_date)
CPI.index.name = 'Date'

## 2.6. 연방 기금 금리 (Federal funds rate)
FEDFUNDS = fdr.DataReader('FRED:FEDFUNDS', start_date, end_date)
FEDFUNDS.index.name = 'Date'

## 2.7. 국내총생산 (Gross Domestic Product : GDP)
GDP = pd.DataFrame(fred.get_series('GDP',observation_start=start_date, observation_end = end_date),columns=['GDP'])
GDP.index.name = 'Date'

KeyboardInterrupt: 

In [ ]:
## 보간법과 홀트-윈터스 지수 평활법

# DGS : 미국 국채 수익률
# 데이터의 마지막 날짜까지의 선형 보간
itp_df = DGS.resample('D').asfreq() 
itp_df = itp_df.interpolate(method='linear') 

# 홀트-윈터스 지수 평활법: 제공된 데이터의 마지막 날짜부터 원하는 종료 날짜까지의 예측
stock_end_date = stock_df.index[-1]
forecast_steps = (pd.to_datetime(stock_end_date) - itp_df.index[-1]).days   
forecast_df = pd.DataFrame(index=pd.date_range(itp_df.index[-1] + pd.Timedelta(days=1), stock_end_date)) 

for column in itp_df.columns:
    model = ExponentialSmoothing(itp_df[column], trend='add', seasonal=None).fit() # 홀트-윈터의 지수 평활화
    forecast = model.forecast(steps=forecast_steps)
    forecast_df[column] = forecast

int_DGS = pd.concat([itp_df, forecast_df])

# Unemployment Rate : 실업률
# 데이터의 마지막 날짜까지의 선형 보간
itp_unrate = Unemployment_Rate.resample('D').asfreq()
itp_unrate = itp_unrate.interpolate()

# 홀트-윈터스 지수 평활법: 제공된 데이터의 마지막 날짜부터 원하는 종료 날짜까지의 예측
forecast_steps_unrate = (pd.to_datetime(stock_end_date) - itp_unrate.index[-1]).days 
forecast_df_unrate = pd.DataFrame(index=pd.date_range(itp_unrate.index[-1] + pd.Timedelta(days=1), stock_end_date)) 

model_unrate = ExponentialSmoothing(itp_unrate['UNRATE'], trend='add', seasonal=None).fit()
forecast_unrate = model_unrate.forecast(steps=forecast_steps_unrate)
forecast_df_unrate['UNRATE'] = forecast_unrate

int_Unemployment_Rate = pd.concat([itp_unrate, forecast_df_unrate])

# CPI
# Linear interpolation up to the last date of the data
itp_CPI = CPI.resample('D').asfreq()
itp_CPI = itp_CPI.interpolate()

# Holt-Winters' Exponential Smoothing: Forecasting from the last date in provided data to the desired End Date
forecast_steps_CPI = (pd.to_datetime(stock_end_date) - itp_CPI.index[-1]).days 
forecast_df_CPI = pd.DataFrame(index=pd.date_range(itp_CPI.index[-1] + pd.Timedelta(days=1), stock_end_date)) 

model_unrate = ExponentialSmoothing(itp_CPI['CPIAUCSL'], trend='add', seasonal=None).fit()
forecast_unrate = model_unrate.forecast(steps=forecast_steps_CPI)
forecast_df_CPI['CPIAUCSL'] = forecast_unrate

int_CPI = pd.concat([itp_CPI, forecast_df_CPI])

# FEDFUNDs
# Linear interpolation up to the last date of the data
itp_fedfunds = FEDFUNDS.resample('D').asfreq()
itp_fedfunds = itp_fedfunds.interpolate()

# Holt-Winters' Exponential Smoothing: Forecasting from the last date in provided data to the desired End Date
forecast_steps_fedfunds = (pd.to_datetime(stock_end_date) - itp_fedfunds.index[-1]).days + 1
forecast_df_fedfunds = pd.DataFrame(index=pd.date_range(itp_fedfunds.index[-1] + pd.Timedelta(days=1), stock_end_date)) 

model_fedfunds = ExponentialSmoothing(itp_fedfunds['FEDFUNDS'], trend='add', seasonal=None).fit()
forecast_fedfunds = model_fedfunds.forecast(steps=forecast_steps_fedfunds)
forecast_df_fedfunds['FEDFUNDS'] = forecast_fedfunds

itp_FEDFUNDS = pd.concat([itp_fedfunds, forecast_df_fedfunds])

# GDP
# Linear interpolation up to the last date of the data
itp_gdp = GDP.resample('D').asfreq()
itp_gdp = itp_gdp.interpolate()

# Holt-Winters' Exponential Smoothing: Forecasting from the last date in provided data to the desired End Date
forecast_steps_gdp = (pd.to_datetime(stock_end_date) - itp_gdp.index[-1]).days 
forecast_df_gdp = pd.DataFrame(index=pd.date_range(itp_gdp.index[-1] + pd.Timedelta(days=1), stock_end_date)) 

model_gdp = ExponentialSmoothing(itp_gdp['GDP'], trend='add', seasonal='add', seasonal_periods=4).fit()
forecast_gdp = model_gdp.forecast(steps=forecast_steps_gdp)
forecast_df_gdp['GDP'] = forecast_gdp

itp_GDP = pd.concat([itp_gdp, forecast_df_gdp])

econ_df = adj_close_df.join([int_DGS, T10Y2Y, VIX, int_Unemployment_Rate, int_CPI, itp_FEDFUNDS, itp_GDP ], how='left')

econ_df

### 3. 산업지표 불러오기

In [ ]:
from pandas_datareader import data as pdr

In [ ]:
## 3.1. 주요 주가 지수 불러오기
df = yf.download("^DJI", start=start_date, end=end_date)  # Dow Jones
df1 = yf.download("NDAQ", start=start_date, end=end_date)  # NASDAQ
df2 = yf.download("^SPX", start=start_date, end=end_date)  # S&P500
df3 = yf.download("^RUT", start=start_date, end=end_date)  # Russell 2000

# 열 이름 변경
df.rename(columns={'Adj Close': 'DJI Adj Close', 'Volume': 'DJI Volume'}, inplace=True) 
df1.rename(columns={'Adj Close': 'NDAQ Adj Close', 'Volume': 'NDAQ Volume'}, inplace=True)
df2.rename(columns={'Adj Close': 'SPX Adj Close', 'Volume': 'SPX Volume'}, inplace=True)
df3.rename(columns={'Adj Close': 'RUT Adj Close', 'Volume': 'RUT Volume'}, inplace=True)

# 열 선택
df = df[['DJI Adj Close', 'DJI Volume']] 
df1 = df1[['NDAQ Adj Close', 'NDAQ Volume']]
df2 = df2[['SPX Adj Close', 'SPX Volume']]
df3 = df3[['RUT Adj Close', 'RUT Volume']]

Index_data = pd.concat([df, df1, df2, df3], axis=1, join='outer')
stock_df = stock_df.reset_index()
Index_data = Index_data.reset_index()
Index_data = Index_data.join(stock_df[['Date', 'Adj Close']].set_index('Date'), on='Date', how='left')

## 3.2. ETF 기반 산업 섹터 데이터 (Industry Sector data based on ETF)
sectors = {
    "VDE": "Energy",                 # Vanguard Energy ETF
    "MXI": "Materials",              # iShares Global Materials ETF
    "VIS": "Industrials",            # Vanguard Industrials ETF
    "VCR": "Consumer Cyclical",      # Vanguard Consumer Discretionary ETF
    "XLP": "Consumer Staples",       # Consumer Staples Select Sector SPDR Fund
    "VHT": "Health Care",            # Vanguard Healthcare ETF
    "XLF": "Financials",             # Financial Select Sector SPDR Fund
    "VGT": "Information Technology", # Vanguard Information Technology ETF
    "VOX": "Communication Services", # Vanguard Communication Services ETF
    "XLU": "Utilities",              # Utilities Select Sector SPDR Fund
    "VNQ": "Real Estate"             # Vanguard Real Estate Index Fund
}
sector_data = {}

# Load Adj Close and Volumes for the ETF industry sector
for sector, sector_name in sectors.items():
    data = yf.download(sector, start=start_date, end=end_date)
    data.rename(columns={
        'Adj Close': f'{sector_name} Adj Close',
        'Volume': f'{sector_name} Volume'
    }, inplace=True) 
    sector_data[sector] = data[[f'{sector_name} Adj Close', f'{sector_name} Volume']]
    
ETF = pd.concat(sector_data.values(), axis=1)

# Matching the adjusted closing prices and trading volumes for the ETF industry sector associated with the entered ticker's sector
sector = yf.Ticker(ticker).info.get('sector', None)
sector_columns = [col for col in ETF.columns if sector in col] 
sector_df = ETF[sector_columns]

# Merge into industry data
Industry_df = Index_data.merge(sector_df, on="Date", how="inner")

In [ ]:
Industry_df

### 4. 기업 재무재표 불러오기

In [ ]:
import requests
from bs4 import BeautifulSoup

In [ ]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/89.0.4389.82 Safari/537.36'
}

## 4.1. INCOME STATEMENT
url = f"https://stockanalysis.com/stocks/{ticker}/financials/?p=quarterly" 
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.content, 'html.parser')
element_tables = soup.select("table[data-test='financials']")

Income_df = pd.read_html(str(element_tables))[0] 
Income_df.to_csv(ticker+'.csv', index=False)

FS_Income = Income_df.transpose()
FS_Income.columns = FS_Income.iloc[0]
FS_Income = Income_df.set_index("Quarter Ended").transpose()
FS_Income.index.name = "Date"
FS_Income.to_csv(ticker+'.csv', index=True, encoding='euc-kr')
FS_Income = FS_Income.iloc[:-1, :]

# Convert string to numeric type
for column in FS_Income.columns:
    if FS_Income[column].dtype == 'object':
        FS_Income[column] = FS_Income[column].apply(lambda x: float(x) if '-' in x and x[1:].isdigit() else x)  # Converting strings with numbers after '-'
        if FS_Income[column].dtype == 'object' and FS_Income[column].str.contains('%').any():
            FS_Income[column] = FS_Income[column].apply(lambda x: float(x.replace('%', '')) / 100 if '%' in x else x)   # Converting strings with a percentage sign
        FS_Income[column] = pd.to_numeric(FS_Income[column], errors='coerce')   # Converting other strings to numbers
        
## 4.2. RATIO STATEMENT
url = f"https://stockanalysis.com/stocks/{ticker}/financials/ratios/?p=quarterly" 
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.content, 'html.parser')
element_tables = soup.select("table[data-test='financials']")

Ratio_df = pd.read_html(str(element_tables))[0]
Ratio_df.to_csv(ticker+'.csv', index=False)

FS_Ratio = Ratio_df.transpose()
FS_Ratio.columns = FS_Ratio.iloc[0]
FS_Ratio = Ratio_df.set_index("Quarter Ended").transpose()
FS_Ratio.index.name = "Date"
FS_Ratio.to_csv(ticker+'.csv', index=True, encoding='euc-kr')
FS_Ratio = FS_Ratio.iloc[1:-1, :]

# Convert string to numeric type
for column in FS_Ratio.columns:
    if FS_Ratio[column].dtype == 'object':       
        FS_Ratio[column] = FS_Ratio[column].apply(lambda x: float(x) if '-' in x and x[1:].isdigit() else x)    # Converting strings with numbers after '-'
        if FS_Ratio[column].dtype == 'object' and FS_Ratio[column].str.contains('%').any():
            FS_Ratio[column] = FS_Ratio[column].apply(lambda x: float(x.replace('%', '')) / 100 if '%' in x else x) # Converting strings with a percentage sign
        FS_Ratio[column] = pd.to_numeric(FS_Ratio[column], errors='coerce') # Converting other strings to numbers

## 4.3. Balance Sheet
url = f"https://stockanalysis.com/stocks/{ticker}/financials/balance-sheet/?p=quarterly" 
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.content, 'html.parser')
element_tables = soup.select("table[data-test='financials']")

Balance_df = pd.read_html(str(element_tables))[0] 
Balance_df.to_csv(ticker+'.csv', index=False)

FS_Balance = Balance_df.transpose()
FS_Balance.columns = FS_Balance.iloc[0]
FS_Balance = Balance_df.set_index("Quarter Ended").transpose()
FS_Balance.index.name = "Date"
FS_Balance.to_csv(ticker+'.csv', index=True, encoding='euc-kr')
FS_Balance = FS_Balance.iloc[:-1, :]

# Convert string to numeric type
for column in FS_Balance.columns:
    if FS_Balance[column].dtype == 'object':       
        FS_Balance[column] = FS_Balance[column].apply(lambda x: float(x) if '-' in x and x[1:].isdigit() else x)    # Converting strings with numbers after '-'
        if FS_Balance[column].dtype == 'object' and FS_Balance[column].str.contains('%').any():
            FS_Balance[column] = FS_Balance[column].apply(lambda x: float(x.replace('%', '')) / 100 if '%' in x else x) # Converting strings with a percentage sign
        FS_Balance[column] = pd.to_numeric(FS_Balance[column], errors='coerce') # Converting other strings to numbers

## 4.4. Cash Flow
url = f"https://stockanalysis.com/stocks/{ticker}/financials/cash-flow-statement/?p=quarterly" 
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.content, 'html.parser')
element_tables = soup.select("table[data-test='financials']")

Cash_df = pd.read_html(str(element_tables))[0] 
Cash_df.to_csv(ticker+'.csv', index=False)

FS_Cash = Cash_df.transpose()
FS_Cash.columns = FS_Cash.iloc[0]
FS_Cash = Cash_df.set_index("Quarter Ended").transpose()
FS_Cash.index.name = "Date"
FS_Cash.to_csv(ticker+'.csv', index=True, encoding='euc-kr')
FS_Cash = FS_Cash.iloc[:-1, :]

# Convert string to numeric type
for column in FS_Cash.columns:
    if FS_Cash[column].dtype == 'object':       
        FS_Cash[column] = FS_Cash[column].apply(lambda x: float(x) if '-' in x and x[1:].isdigit() else x)  # Converting strings with numbers after '-'
        if FS_Cash[column].dtype == 'object' and FS_Cash[column].str.contains('%').any():
            FS_Cash[column] = FS_Cash[column].apply(lambda x: float(x.replace('%', '')) / 100 if '%' in x else x)   # Converting strings with a percentage sign       
        FS_Cash[column] = pd.to_numeric(FS_Cash[column], errors='coerce')   # Converting other strings to numbers

In [ ]:
# Add ROE (Return on Equity)
FS_Ratio['ROE'] = FS_Income['Net Income'] / FS_Balance['Shareholders\' Equity']

# Merge all financial statement data
FS_Summary = pd.concat([FS_Income, FS_Balance, FS_Ratio, FS_Cash], axis=1)
FS_Summary.index = pd.to_datetime(FS_Summary.index)
duplicated_columns = FS_Summary.columns[FS_Summary.columns.duplicated()].unique()
FS_Summary = FS_Summary.drop(columns=duplicated_columns)

## Interpolation and Holt-Winters' Exponential Smoothing for Monthly Data

# Linear interpolation up to the last date of the data
itp_df = FS_Summary.resample('D').asfreq() 
for column in itp_df.columns:
    itp_df[column] = itp_df[column].interpolate(method='linear') # using linear method
    
# Holt-Winters' Exponential Smoothing: Forecasting from the last date in provided data to the desired End Date
forecast_steps = (pd.to_datetime(stock_end_date) - itp_df.index[-1]).days
forecast_df = pd.DataFrame(index=pd.date_range(itp_df.index[-1] + pd.Timedelta(days=1), stock_end_date)) 

for column in itp_df.columns:
    model = ExponentialSmoothing(itp_df[column], trend='add', seasonal= None, seasonal_periods=4).fit() 
    forecast = model.forecast(steps=forecast_steps)
    forecast_df[column] = forecast

daily_FS_Summary = pd.concat([itp_df, forecast_df])
daily_FS_Summary = daily_FS_Summary.dropna(axis=1, how='any')

# Add Adj Close
daily_FS_Summary = daily_FS_Summary.merge(adj_close_df, left_index=True, right_index=True, how='left')
daily_FS_Summary = daily_FS_Summary.dropna()
daily_FS_Summary['Date'] = daily_FS_Summary.index
daily_FS_Summary = daily_FS_Summary.reset_index(drop=True)
daily_FS_Summary = daily_FS_Summary.set_index('Date').sort_index()

# Filter features with a correlation coefficient of 0.9 or higher
correlation = daily_FS_Summary.corr()['Adj Close']
selected_features = correlation[correlation.abs() > 0.9].index.tolist() 
daily_FS_Summary= daily_FS_Summary[selected_features]

daily_FS_Summary

In [ ]:
# 1. Economic Indicators plot comparison
econ_month_df = Unemployment_Rate.join([CPI, FEDFUNDS, GDP], how='left')
fig, axes = plt.subplots(2, 2, figsize=(15, 10))  
axes = axes.ravel()

for idx, column in enumerate(econ_month_df.columns):
    ax = axes[idx]
    econ_month_df[column].plot(ax=ax, label='Original', linestyle='-', color='blue', marker='o', ms=3)
    econ_df[column].plot(ax=ax, label='Forecast', linestyle='--', color='red')
    ax.set_title(column)
    ax.legend(loc='best')
    ax.grid(True)
plt.tight_layout()
plt.suptitle('Economic indicator Comparison of Actual and Interpolated Graphs', y=1.05)
plt.show()

# 2. Financial Statement Indicators plot comparison
common_columns = FS_Summary.columns.intersection(daily_FS_Summary.columns)
common_columns
page_size = 9
total_pages = -(-len(common_columns) // page_size)

for page in range(total_pages):
    fig, axes = plt.subplots(3, 3, figsize=(15, 10))
    axes = axes.ravel() 
    end_idx = min((page + 1) * page_size, len(common_columns))    
    for idx in range(page * page_size, end_idx):
        column = common_columns[idx]
        FS_Summary[column].plot(ax=axes[idx % page_size], label='Original', linestyle='-', color='blue')
        daily_FS_Summary[column].plot(ax=axes[idx % page_size], label='Forecast', linestyle='--', color='red')
        axes[idx % page_size].set_title(column)
        axes[idx % page_size].legend(loc='best')
        axes[idx % page_size].grid(True)
    plt.tight_layout()
    plt.suptitle('Financial Statement indicator Comparison of Actual and Interpolated Graphs', y=1.05)
    plt.show()

### 5. 변수들과 수정종가간의 상관관계 시각화 및 변수 선택

In [ ]:
### Technical Feature Selection

# 1.1. Filter features with a correlation coefficient of 0.9 or higher
correlation = stock_df.corr()['Adj Close']
selected_features = correlation[correlation.abs() > 0.9].index.tolist() 
selected_features2 = selected_features.copy()
selected_features2.remove('Adj Close')
print(selected_features) 

# 1.2. Correlation Graph against Adj Close
ncols = 6  
nrows = 3
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(15, 2*nrows))
for ax, feature in zip(axes.ravel(), selected_features2):
    ax.scatter(stock_df['Adj Close'], stock_df[feature], s=8) 
    ax.set_title(f'Adj Close vs {feature}', fontsize=8)
    ax.set_xlabel('Adj Close', fontsize=8)
    ax.set_ylabel(feature, fontsize=8)
plt.tight_layout()
plt.show()

# 1.3. Select features based on high correlation
tech_df = stock_df[selected_features]


### Fundamental Feature Selection

## Economic Variable Selection
# 2.1. Filter features with a correlation coefficient of 0.8 or higher
correlation = econ_df.corr()['Adj Close']
selected_features = correlation[correlation.abs() > 0.8].index.tolist() 
selected_features2 = selected_features.copy()
selected_features2.remove('Adj Close')
print(selected_features) 

# 2.2. Correlation Graph against Adj Close
ncols = 2  
nrows = 1
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(7, 2*nrows))
for ax, feature in zip(axes.ravel(), selected_features2):
    ax.scatter(econ_df['Adj Close'], econ_df[feature], s=8) 
    ax.set_title(f'Adj Close vs {feature}', fontsize=8)
    ax.set_xlabel('Adj Close', fontsize=8)
    ax.set_ylabel(feature, fontsize=8)
plt.tight_layout()
plt.show()

# 2.3. Select features based on high correlation
econ_df = econ_df[selected_features]

## Industry Variable Selection
# 3.1. Filter features with a correlation coefficient of 0.9 or higher
correlation = Industry_df.corr()['Adj Close']
selected_features = correlation[correlation.abs() > 0.9].index.tolist() 
selected_features2 = selected_features.copy()
selected_features2.remove('Adj Close')
print(selected_features) 

# 3.2.
ncols = 2  
nrows = 2
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(15, 2*nrows))
for ax, feature in zip(axes.ravel(), selected_features2):
    ax.scatter(Industry_df['Adj Close'], Industry_df[feature], s=8) 
    ax.set_title(f'Adj Close vs {feature}', fontsize=8)
    ax.set_xlabel('Adj Close', fontsize=8)
    ax.set_ylabel(feature, fontsize=8)
plt.tight_layout()
plt.show()

# 3.3. Select features based on high correlation
Industry_df = Industry_df[selected_features]

## Company Variable Selection
# 4.1. Filter features with a correlation coefficient of 0.9 or higher
correlation = daily_FS_Summary.corr()['Adj Close']
selected_features = correlation[correlation.abs() > 0.9].index.tolist() 
selected_features2 = selected_features.copy()
selected_features2.remove('Adj Close')
print(selected_features) 

# 4.2.
ncols = 6  
nrows = 4
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(15, 2*nrows))
for ax, feature in zip(axes.ravel(), selected_features2):
    ax.scatter(daily_FS_Summary['Adj Close'], daily_FS_Summary[feature], s=8) 
    ax.set_title(f'Adj Close vs {feature}', fontsize=8)
    ax.set_xlabel('Adj Close', fontsize=8)
    ax.set_ylabel(feature, fontsize=8)
plt.tight_layout()
plt.show()

### 1. 주가 예측 모델 : 가격 예측

In [ ]:
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from keras.regularizers import L1L2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [ ]:
## 1.1. Load data and Remove Missing value
df = stock_df

df = df.dropna()
df.isnull().sum() 

## 1.2. Apply MinMax Normalization
df = df.set_index('Date')
scaler = MinMaxScaler()
scale_cols = df.columns.tolist()
scaled_df = scaler.fit_transform(df[scale_cols])
scaled_df = pd.DataFrame(scaled_df, columns=scale_cols)

# Define Input Parameter: feature, label -> numpy type
def make_sequene_dataset(feature, label, window_size):
    feature_list = []      
    label_list = []        
    for i in range(len(feature)-window_size):
        feature_list.append(feature[i:i+window_size]) 
        label_list.append(label[i+window_size]) 
    return np.array(feature_list), np.array(label_list) 

# Create feature_df, label_df 
feature_cols = df.columns.drop('Adj Close').tolist()
label_cols = [ 'Adj Close' ]

feature_df = pd.DataFrame(scaled_df, columns=feature_cols)
label_df = pd.DataFrame(scaled_df, columns=label_cols)

# DataFrame -> Numpy transform
feature_np = feature_df.to_numpy()
label_np = label_df.to_numpy()

print(f'feature_np.shape:{feature_np.shape}')
print(f'label_np.shape:{label_np.shape}')

In [ ]:
## 1.3. Create data    
# Set window size
window_size = 30
X, Y = make_sequene_dataset(feature_np, label_np, window_size)

# 3.2. Split into train, test (8:2)
split = int(len(X)*0.80) 
x_train = X[0:split]
y_train = Y[0:split]

x_test = X[split:]
y_test = Y[split:]

print(f'X.shape:{X.shape}, Y.shape:{Y.shape}')
print(f'x_train.shape:{x_train.shape}, y_train.shape:{y_train.shape}')
print(f'x_test.shape:{x_test.shape}, y_test.shape:{y_test.shape}') 

In [ ]:
## 1.4. Construct and Compile model
model = Sequential()

model.add(LSTM(64, activation='tanh', input_shape=x_train[0].shape, return_sequences=False, 
               kernel_regularizer=L1L2(l1=0.0001, l2=0.0001), recurrent_regularizer=L1L2(l1=0.0001, l2=0.0001)))
model.add(Dropout(0.2))

model.add(Dense(1, activation='linear'))

model.compile(loss='mse', optimizer='adam', metrics=['mae'])

model.summary()

# List to record the loss values during the model training process
train_loss_history = []
val_loss_history = []ㅇ

# Train model (Apply earlystopping)
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=20)

hist = model.fit(x_train, y_train, 
          validation_data=(x_test, y_test),
          epochs=1000, batch_size=150,       
          callbacks=[early_stop]) 

pred = model.predict(x_test)

In [ ]:
# Evaluation 1: Prediction Graph
plt.figure(figsize=(12, 6))
plt.title(f'{ticker} Technical analysis Prediction model')
plt.ylabel('Adj Close')
plt.xlabel('period')
plt.plot(y_test, label='actual')
plt.plot(pred, label='prediction')
plt.grid()
plt.legend(loc='best')
plt.show()

# Evaluation 2: Learning Curve
train_loss_history.extend(hist.history['loss']) 
val_loss_history.extend(hist.history['val_loss'])

plt.figure(figsize=(12, 6))
plt.plot(train_loss_history, label='Training Loss')
plt.plot(val_loss_history, label='Validation Loss')
plt.legend()
plt.title('Loss History')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.show()

# Evaluation 2: MAPE, MAE, RMSE
mape = np.sum(abs(y_test - pred) / y_test) / len(x_test)
mae = np.mean(np.abs(y_test - pred))
rmse = np.sqrt(np.mean(np.square(y_test - pred)))

metrics_df = pd.DataFrame({
    'Metrics': ['MAPE', 'MAE', 'RMSE'],
    'Values': [mape, mae, rmse]})

print(metrics_df)

### 2. 주가 예측 모델 : 업 다운 이진 분류

In [31]:
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.regularizers import L1L2
import tensorflow as tf
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV

In [23]:
# 데이터 전처리
df = stock_df.dropna()

# MinMax Normalization
scaler = MinMaxScaler()
scale_cols = df.columns.tolist()
scaled_df = scaler.fit_transform(df[scale_cols])
scaled_df = pd.DataFrame(scaled_df, columns=scale_cols)

# 다음날 주가가 오를지 내릴지를 라벨로 정의
df['Target'] = (df['Adj Close'].shift(-1) > df['Adj Close']).astype(int)

C:\TempFolder\ipykernel_12044\2187773920.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Target'] = (df['Adj Close'].shift(-1) > df['Adj Close']).astype(int)


In [24]:
# numpy type으로 변경
def make_sequene_dataset(feature, label, window_size):
    feature_list = []      
    label_list = []        
    for i in range(len(feature)-window_size):
        feature_list.append(feature[i:i+window_size]) 
        label_list.append(label[i+window_size]) 
    return np.array(feature_list), np.array(label_list)

# feature_df, label_df 생성
feature_cols = df.columns.drop('Target').tolist()
label_cols = ['Target']

feature_df = pd.DataFrame(scaled_df, columns=feature_cols)
label_df = df[label_cols]  # 원본 데이터에서 레이블 설정

# DataFrame -> Numpy transform
feature_np = feature_df.to_numpy()
label_np = label_df.to_numpy()

print(f'feature_np.shape:{feature_np.shape}')
print(f'label_np.shape:{label_np.shape}')

feature_np.shape:(1074, 23)
label_np.shape:(1074, 1)


In [25]:
# Set window size
window_size = 30
X, Y = make_sequene_dataset(feature_np, label_np, window_size)

# Split into train, test (8:2)
split = int(len(X) * 0.80) 
x_train = X[0:split]
y_train = Y[0:split]

x_test = X[split:]
y_test = Y[split:]

print(f'X.shape:{X.shape}, Y.shape:{Y.shape}')
print(f'x_train.shape:{x_train.shape}, y_train.shape:{y_train.shape}')
print(f'x_test.shape:{x_test.shape}, y_test.shape:{y_test.shape}') 

X.shape:(1044, 30, 23), Y.shape:(1044, 1)
x_train.shape:(835, 30, 23), y_train.shape:(835, 1)
x_test.shape:(209, 30, 23), y_test.shape:(209, 1)


In [34]:
# 모델 생성 함수
def create_model(units=64, dropout_rate=0.2, optimizer='adam'):
    model = Sequential()
    model.add(LSTM(units, activation='tanh', 
                   input_shape=x_train[0].shape, 
                   return_sequences=False, 
                   kernel_regularizer=L1L2(l1=0.0001, l2=0.0001), 
                   recurrent_regularizer=L1L2(l1=0.0001, l2=0.0001)))
    model.add(Dropout(dropout_rate))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    return model

# KerasClassifier 래퍼
model = KerasClassifier(model=create_model, verbose=0, units=64, dropout_rate=0.2, optimizer='adam', epochs=50, batch_size=50)

# 하이퍼파라미터 그리드 설정
param_grid = {
    'units': [32, 64, 128],
    'dropout_rate': [0.2, 0.3, 0.4],
    'batch_size': [50, 100, 150],
    'epochs': [50, 100, 200],
    'optimizer': ['adam', 'rmsprop']
}

# GridSearchCV 설정
grid = GridSearchCV(estimator=model, param_grid=param_grid, scoring='accuracy', cv=3)

# 그리드 서치 수행
grid_result = grid.fit(x_train, y_train)

# 최적의 하이퍼파라미터 출력
print(f'Best parameters: {grid_result.best_params_}')
print(f'Best score: {grid_result.best_score_:.4f}')

c:\Program Files\Python310\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Program Files\Python310\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Program Files\Python310\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


c:\Program Files\Python310\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


c:\Program Files\Python310\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Program Files\Python310\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Program Files\Python310\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Program Files\Python310\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_

KeyboardInterrupt: 

In [ ]:
# 모델 구성 및 컴파일
model = Sequential()

model.add(LSTM(64, activation='tanh', 
               input_shape=x_train[0].shape, 
               return_sequences=False, 
               kernel_regularizer=L1L2(l1=0.0001, l2=0.0001), 
               recurrent_regularizer=L1L2(l1=0.0001, l2=0.0001)))

model.add(Dropout(0.2)) # 비활성화 뉴련율
model.add(Dense(1, activation='sigmoid'))  # 이진 분류이므로 sigmoid 활성화 함수 사용

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()

# 모델 훈련
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=20)

hist = model.fit(x_train, y_train, 
                 validation_data=(x_test, y_test),
                 epochs=1000, batch_size=150,       
                 callbacks=[early_stop]) 

# 예측
pred = (model.predict(x_test) > 0.5).astype(int)  # 0.5를 기준으로 상승(1)과 하락(0) 예측

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# 모델 예측
y_pred = (model.predict(x_test) > 0.5).astype(int)  # 0.5를 기준으로 상승(1)과 하락(0) 예측

# 정확도 계산
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

In [ ]:
# 혼동행렬 계산
conf_matrix = confusion_matrix(y_test, y_pred)

# 혼동행렬 시각화
plt.figure(figsize=(6, 4))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted Down', 'Predicted Up'], yticklabels=['Actual Down', 'Actual Up'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()